# Task 3.7: Error Analysis

**Project:** Potato Quality Classification (Fresh vs. Rotten)  
**ILO:** 8.2 E  

This notebook performs error analysis on the best-performing model: **CNN v4** (MobileNetV2 frozen, 224x224, 97.05% test accuracy). The goal is to understand where and why the model fails, categorize the errors, and propose improvements.

### Steps
1. Rebuild and load the best model (CNN v4)
2. Identify all misclassified images
3. Visualize the errors with true label, predicted label, and confidence
4. Categorize errors by type
5. Quantitative analysis of error categories
6. Propose improvements

---
## 1. Imports & Setup

In [ ]:
!pip install seaborn scikit-learn -q

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense, Dropout, GlobalAveragePooling2D, BatchNormalization
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2

from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score, f1_score
)

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('TensorFlow version:', tf.__version__)

---
## 2. Load Data & Saved Model

We load the saved model from Iteration 4 (`cnn_v4_best.keras`) to ensure we analyse the exact same model that achieved the best results.

In [ ]:
BASE_PATH  = 'data/processed'
TRAIN_PATH = os.path.join(BASE_PATH, 'train')
VAL_PATH   = os.path.join(BASE_PATH, 'val')
TEST_PATH  = os.path.join(BASE_PATH, 'test')

IMG_SIZE   = (224, 224)
BATCH_SIZE = 32

import shutil, glob
for cp in glob.glob(os.path.join(BASE_PATH, '*', '.ipynb_checkpoints')):
    shutil.rmtree(cp)
    print(f'Removed: {cp}')

datagen = ImageDataGenerator(rescale=1.0 / 255)

train_gen = datagen.flow_from_directory(
    TRAIN_PATH, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=True, seed=SEED
)

val_gen = datagen.flow_from_directory(
    VAL_PATH, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False
)

test_gen = datagen.flow_from_directory(
    TEST_PATH, target_size=IMG_SIZE, batch_size=1,
    class_mode='categorical', shuffle=False
)

CLASS_NAMES = list(train_gen.class_indices.keys())
assert len(train_gen.class_indices) == 2
print('Classes:', train_gen.class_indices)
print('Test samples:', test_gen.samples)

In [ ]:
# -- Load the saved best model from Iteration 4 --
from tensorflow.keras.models import load_model
import os

model_path = os.path.join('data', 'cnn_v4_best.keras')
model = load_model(model_path)
print(f'Model loaded from: {os.path.abspath(model_path)}')

# -- Evaluate on test set --
test_gen.reset()
test_loss, test_acc = model.evaluate(test_gen, verbose=0)
print(f'Test Accuracy: {test_acc * 100:.2f}%')

---
## 3. Identify Misclassified Images

We predict on every test image individually and collect the ones where the prediction does not match the true label.

In [ ]:
test_gen.reset()
y_pred_proba = model.predict(test_gen, verbose=0)
y_pred = np.argmax(y_pred_proba, axis=1)
y_true = test_gen.classes
y_conf = np.max(y_pred_proba, axis=1)  # confidence of prediction

file_paths = test_gen.filepaths

misclassified_idx = np.where(y_pred != y_true)[0]

print(f'Total test images: {len(y_true)}')
print(f'Correct: {len(y_true) - len(misclassified_idx)}')
print(f'Misclassified: {len(misclassified_idx)}')
print(f'Accuracy: {(1 - len(misclassified_idx)/len(y_true))*100:.2f}%')

errors = []
for idx in misclassified_idx:
    errors.append({
        'index': idx,
        'file': os.path.basename(file_paths[idx]),
        'true_label': CLASS_NAMES[y_true[idx]],
        'predicted': CLASS_NAMES[y_pred[idx]],
        'confidence': f'{y_conf[idx]*100:.1f}%',
        'path': file_paths[idx],
    })

import pandas as pd
errors_df = pd.DataFrame(errors)
print('\nMisclassified images:')
errors_df[['file', 'true_label', 'predicted', 'confidence']]

---
## 4. Visualize Misclassified Images

Below we show every misclassified image with its true label, predicted label, and the model's confidence score. This helps identify visual patterns in the errors.

In [ ]:
n_errors = len(misclassified_idx)
cols = 4
rows = max(1, (n_errors + cols - 1) // cols)

fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4.5 * rows))
fig.suptitle(f'All Misclassified Images ({n_errors} errors)', fontsize=16, fontweight='bold')

if rows == 1:
    axes = np.array([axes])

for i, ax in enumerate(axes.flat):
    if i < n_errors:
        idx = misclassified_idx[i]
        img = plt.imread(file_paths[idx])
        ax.imshow(img)
        
        true_lbl = CLASS_NAMES[y_true[idx]]
        pred_lbl = CLASS_NAMES[y_pred[idx]]
        conf = y_conf[idx] * 100
        
        ax.set_title(
            f'True: {true_lbl}\nPred: {pred_lbl} ({conf:.1f}%)',
            fontsize=9, color='red', fontweight='bold'
        )

        ax.set_xlabel(os.path.basename(file_paths[idx]), fontsize=7)
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.show()

---
## 5. Error Type Analysis

After visually inspecting the misclassified images above, we categorize each error by its likely cause. Common error types for potato classification include:

| Error Type | Description |
|---|---|
| **Ambiguous decay** | Potato shows very early/subtle signs of rot that could be mistaken for natural skin variation |
| **Lighting/color issue** | Poor lighting, shadows, or color cast makes the potato appear different than it is |
| **Background confusion** | Background color or objects interfere with classification |
| **Unusual variety** | Potato variety with atypical skin color (e.g., red, purple) confused with decay |
| **Image quality** | Blurry, low resolution, or badly cropped image |
| **Possible mislabel** | Image appears to be labeled incorrectly in the dataset |

In [ ]:
# ============================================================
# MANUAL STEP: Categorize each error after visual inspection
# ============================================================
# Look at the misclassified images above and assign each one
# to a category. Replace the placeholder categories below.
#
# Categories:
#   'ambiguous'   - subtle/early decay, hard to tell
#   'lighting'    - lighting or color issue
#   'background'  - background interferes
#   'variety'     - unusual potato variety or composition
#   'quality'     - blurry or low quality image
#   'mislabel'    - appears incorrectly labeled
#
# NOTE: You must fill in the categories manually based on
# your visual inspection of the images above. The number of
# categories must match the number of misclassified images.
# ============================================================

error_categories = [
    'variety',     # fresh_0530 - cut potato, unusual image
    'quality',     # fresh_0977 - dark background, blurry
    'quality',     # fresh_1103 - dark background, poor lighting
    'quality',     # fresh_1134 - dark background, poor lighting
    'variety',     # fresh_1309 - multiple potatoes in frame
    'mislabel',    # fresh_1667 - looks rotten, possibly mislabeled
    'ambiguous',   # rotten_0298 - subtle rot, looks fresh
    'variety',     # rotten_0414 - sprouted potato with watermark
    'variety',     # rotten_0439 - sprouted potatoes, not typical rot
    'mislabel',    # rotten_0451 - pile of fresh potatoes, likely mislabeled
]

if len(error_categories) != len(errors_df):
    print(f'⚠ Please categorize all {len(errors_df)} errors above.')
    print(f'  Currently {len(error_categories)} categories assigned.')
    print('  Using "uncategorized" as placeholder for now.\n')
    error_categories = ['uncategorized'] * len(errors_df)

errors_df['category'] = error_categories
errors_df[['file', 'true_label', 'predicted', 'confidence', 'category']]

---
## 6. Quantitative Error Analysis

We count how many errors fall into each category and calculate the percentage distribution.

In [ ]:
# -- Error category distribution --
cat_counts = errors_df['category'].value_counts()
cat_pct = (cat_counts / len(errors_df) * 100).round(1)

print('Error Category Distribution:')
print('=' * 45)
for cat, count in cat_counts.items():
    pct = cat_pct[cat]
    print(f'  {cat:20s}: {count:3d} errors ({pct:.1f}%)')
print(f'  {"TOTAL":20s}: {len(errors_df):3d} errors')

# -- Bar chart --
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(cat_counts.index, cat_counts.values, color='#e74c3c', edgecolor='black')

for bar, pct in zip(bars, cat_pct.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.2,
        f'{pct:.1f}%',
        ha='center', va='bottom', fontsize=11, fontweight='bold'
    )

ax.set_ylabel('Number of Errors', fontsize=12)
ax.set_title('Error Distribution by Category', fontsize=13)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
print('Error Direction:')
print('=' * 45)
fresh_as_rotten = sum((errors_df['true_label'] == 'fresh') & (errors_df['predicted'] == 'rotten'))
rotten_as_fresh = sum((errors_df['true_label'] == 'rotten') & (errors_df['predicted'] == 'fresh'))
print(f'  Fresh predicted as Rotten: {fresh_as_rotten}')
print(f'  Rotten predicted as Fresh: {rotten_as_fresh}')

print(f'\nConfidence on misclassified images:')
print(f'  Mean confidence: {y_conf[misclassified_idx].mean()*100:.1f}%')
print(f'  Min confidence:  {y_conf[misclassified_idx].min()*100:.1f}%')
print(f'  Max confidence:  {y_conf[misclassified_idx].max()*100:.1f}%')

if y_conf[misclassified_idx].mean() > 0.8:
    print('\n  → High confidence on wrong predictions suggests these images')
    print('    are genuinely ambiguous or possibly mislabeled.')
else:
    print('\n  → Low confidence on wrong predictions suggests the model')
    print('    is uncertain about these images, which is expected for borderline cases.')

---
## 7. Confusion Matrix (Final Model)

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES
)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix - CNN v4 (Best Model)')
plt.tight_layout()
plt.show()

print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

---
## 8. Proposed Improvements

Based on the error analysis, the following improvements could address the identified error patterns:

**1. Unusual variety/composition (40% of errors - 4 out of 10)**
The largest error category consists of images that differ from typical single-potato training examples: cut potatoes (fresh_0530), images with multiple potatoes in one frame (fresh_1309), and sprouted potatoes (rotten_0414, rotten_0439). To address this:
- Include more diverse training images showing cut potatoes, sprouted potatoes, and different potato varieties
- Remove or relabel images containing multiple potatoes, as these do not match the single-potato classification task that FreshScan is designed for

**2. Image quality issues (30% of errors - 3 out of 10)**
Three errors were caused by poor image quality: dark backgrounds with poor lighting conditions (fresh_0977, fresh_1103, fresh_1134). All three were fresh potatoes that appeared darker than usual due to the photography conditions. To address this:
- Apply brightness and contrast normalisation as a preprocessing step
- Filter out or re-photograph very dark or low quality images from the training set
- In the FreshScan deployment, ensure consistent lighting conditions on the production line

**3. Possible mislabels (20% of errors - 2 out of 10)**
Two images appear to be incorrectly labeled in the dataset. Fresh_1667 looks like a rotten potato but is labeled as fresh, and rotten_0451 shows a pile of fresh-looking potatoes but is labeled as rotten. The model predicted these with high confidence in the opposite direction (95.5% and 63.8%). To address this:
- Review the dataset for similar mislabeled images and correct them
- Re-train the model with corrected labels, which could further improve accuracy

**4. Ambiguous cases (10% of errors - 1 out of 10)**
One image (rotten_0298) shows a potato with very subtle signs of decay that is difficult to distinguish from a fresh potato, even for human observers. To address this:
- Collect more training images of potatoes in early stages of decay to help the model learn the subtle visual differences
- Consider adding a third class ("borderline") for potatoes that are neither clearly fresh nor clearly rotten